# ÉTAPE 01 : ENTRAÎNEMENT DU MODÈLE

Ce script est exécuté AVANT le démarrage de l'API.

Objectif :
    Dataset → séparation train/test → pipeline → entraînement
            → évaluation → sauvegarde de model.pkl

Pourquoi ?
    L'API ne doit pas entraîner le modèle à chaque requête.
    Elle charge une fois le modèle déjà entraîné.

In [1]:
from pathlib import Path

import joblib
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [2]:
BASE_DIR = Path.cwd()
DATA_PATH = BASE_DIR / "data" / "houses.csv"
MODEL_PATH = BASE_DIR / "model" / "model.pkl"

# ÉTAPE 02 : Charger les données
Le CSV contient trois variables explicatives et une cible : surface, bedrooms, age → price.

In [3]:
df = pd.read_csv(DATA_PATH)
df

,surface,bedrooms,age,price
0,50,1,20,95000
1,60,1,15,115000
2,70,2,15,135000
3,80,2,12,155000
4,90,2,10,175000
5,100,3,10,195000
6,110,3,8,220000
7,120,3,7,245000
8,130,3,5,275000
9,140,4,6,295000


In [4]:
X = df[["surface", "bedrooms", "age"]]
y = df["price"]

# ÉTAPE 03 : Séparer les données
Une partie sert à apprendre et une autre à vérifier le comportement du modèle sur des données qu'il n'a pas utilisées pendant l'apprentissage.

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

# ÉTAPE 04 : Définir le preprocessing
StandardScaler transforme les variables afin qu'elles soient sur une échelle comparable. Ici, toutes les variables sont numériques.

In [6]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            StandardScaler(),
            ["surface", "bedrooms", "age"],
        )
    ]
)

# ÉTAPE 05 : Construire une Pipeline
Preprocessing + Modèle sont sauvegardés ensemble.

In [7]:
pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LinearRegression()),
    ]
)

# ÉTAPE 06 : Entraîner

In [8]:
pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('numeric', StandardScaler(),
                                                  ['surface', 'bedrooms',
                                                   'age'])])),
                ('model', LinearRegression())])

# ÉTAPE 07 : Évaluer sur le jeu de test

In [9]:
predictions = pipeline.predict(X_test)

mae = mean_absolute_error(y_test, predictions)
rmse = mean_squared_error(y_test, predictions) ** 0.5
r2 = r2_score(y_test, predictions)

# ÉTAPE 08 : Sauvegarder la pipeline complète
L'API pourra ensuite faire pipeline.predict(...) directement.

In [10]:
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(pipeline, MODEL_PATH)

print("=" * 60)
print("Modèle entraîné avec succès.")
print(f"Model saved to: {MODEL_PATH}")
print("-" * 60)
print(f"MAE  : {mae:,.2f} DZD")
print(f"RMSE : {rmse:,.2f} DZD")
print(f"R²   : {r2:.4f}")
print("=" * 60)

Modèle entraîné avec succès.
Model saved to: /Users/mac/Downloads/Data Science - Bootcamp/Deep Learning NVIDIA certified/08 Déployer un modèle d’IA avec FastAPI et Streamlit/fastapi_ai_project/model/model.pkl
------------------------------------------------------------
MAE  : 11,876.91 DZD
RMSE : 14,497.73 DZD
R²   : 0.9756


# INTERPRÉTATION :
- MAE  → erreur absolue moyenne. Plus elle est faible, mieux c'est.
- RMSE → similaire à MAE mais pénalise davantage les grosses erreurs.
- R²   → proche de 1 signifie que le modèle explique une grande partie de la variabilité de la cible sur les données évaluées.

# ATTENTION :
- Ces données sont pédagogiques et artificielles. Les métriques ne doivent pas être interprétées comme la performance d'un vrai modèle immobilier.

# Réaliser par : OUARAS Khelil Rafik